<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The output is a ranked queue built from my ML-08/ML-09 model: for each page in the honest, held-out (unseen-client) test slice, the model's `decline_probability`, plus a **reason code** derived directly from that page's own top contributing feature (Logistic Regression coefficient x standardized value — not a hand-picked label, the actual reason the model scored this row the way it did). Reason codes observed on this run: `losing_clicks`, `high_visibility_declining`, `scroll_drop`, `aging_content`, `session_decline`, `position_slipping`, `stale_and_unattended`, plus a few thin word/char-count-driven cases. Per `writing-honest-claims`, this is decision-support language throughout — a ranked list worth a human's attention, not a verdict.


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "impression_tier"]

X = df[NUMERIC + CATEGORICAL]
y = df["is_declining_label"]
groups = df["client_id"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=42))]).fit(X.iloc[tr_idx], y.iloc[tr_idx])

# Score only the held-out, never-trained-on clients -- same honest split as ML-08/ML-09
queue = df.iloc[te_idx].copy()
queue["decline_probability"] = pipe.predict_proba(X.iloc[te_idx])[:, 1]

# Per-row reason code: which feature contributed most to THIS row's score
num_pipe = pipe.named_steps["pre"].named_transformers_["num"]
X_num_scaled = num_pipe.named_steps["scale"].transform(num_pipe.named_steps["impute"].transform(X.iloc[te_idx][NUMERIC]))
coefs = pipe.named_steps["clf"].coef_[0][:len(NUMERIC)]
contrib = X_num_scaled * coefs
top_feat_idx = np.argmax(contrib, axis=1)

REASON_MAP = {
    "log_impressions_90d": "high_visibility_declining", "log_clicks_90d": "losing_clicks",
    "content_age_days": "aging_content", "days_since_last_update": "stale_and_unattended",
    "avg_position": "position_slipping", "ctr": "ctr_underperforming",
    "search_volume": "high_demand_at_risk", "engagement_rate": "engagement_drop",
    "scroll_rate": "scroll_drop", "ai_traffic_pct": "ai_traffic_shift",
    "log_sessions_90d": "session_decline", "log_ai_sessions_90d": "ai_session_decline",
    "word_count": "thin_content", "char_count": "thin_content",
    "days_with_impressions": "inconsistent_visibility", "days_with_sessions": "inconsistent_traffic",
    "competition": "competitive_pressure",
}
queue["reason_code"] = [REASON_MAP.get(NUMERIC[i], NUMERIC[i]) for i in top_feat_idx]

# Cost/value proxy (matches the paper's own "clicks x CPC, not impressions x CPC" rule)
queue["click_equivalent_value"] = queue["clicks_90d"] * queue["cpc"]

queue["action"] = np.where(queue["decline_probability"] >= 0.6, "refresh_review", "monitor")

ranked = queue.sort_values(["action", "decline_probability", "click_equivalent_value"], ascending=[True, False, False])
cols = ["content_id", "client_id", "decline_probability", "reason_code", "action",
        "click_equivalent_value", "avg_position", "days_since_last_update", "impressions_90d"]

print(f"queue size: {len(ranked):,} pages (held-out test clients only)")
print(f"flagged refresh_review: {(ranked['action']=='refresh_review').sum():,}")
print()
print(ranked["reason_code"].value_counts())
print()
ranked[cols].head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content editor or SEO lead at a client with a large page library uses this queue as a *starting point* for weekly triage — which pages are most worth a human look this week — not as an automatic action list. The score is decision-support, matching the claim ladder: "these pages look worth reviewing first, because [reason code]," never "these pages will decline" or "refreshing these will fix them."

**Where it stops being valid:**
- **Outside this snapshot.** The model was trained on one 90-day cross-section of 32 clients in the starter sample. It has not been validated on the full warehouse (per ML-04/ML-08), on a different time period, or on a client outside this portfolio's mix of content types.
- **Precision, not certainty.** Even on the honest held-out split, precision@50 was 0.76 — meaning roughly 1 in 4 of the top 50 flagged pages is a false positive. This is a ranking tool, not a binary truth machine.
- **No causal claim.** Per `writing-honest-claims`, nothing here says a refresh *causes* recovery — only that a page's current signals match the pattern associated with decline in this sample. ML-06's own audit already flagged this exact caution in the source paper's Freshness Multiplier finding.
- **Reason codes describe the model's math, not the real-world cause.** `losing_clicks` means "clicks contributed most to this row's score," not "the page's clicks fell for a known reason" — a human still has to look.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Archetype -> action mapping, built from age_tier x freshness_tier (same idea as the
# FlyRank paper's Age-Freshness Matrix, applied to my own held-out queue)
def archetype(row):
    old = row["age_tier"] in ("181-365", "365+")
    stale = row["freshness_tier"] in ("91-180", "181+")
    if old and stale:
        return "decay_zone -> prioritize_refresh"
    if old and not stale:
        return "refreshed_survivor -> protect_and_monitor"
    if not old and stale:
        return "young_but_neglected -> early_review"
    return "growth_engine -> protect_and_extend"

ranked["archetype"] = ranked.apply(archetype, axis=1)
print(ranked["archetype"].value_counts())
print()
print("Cross-tab: archetype vs action (does the queue agree with the age/freshness story?)")
print(pd.crosstab(ranked["archetype"], ranked["action"]))

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `refresh_review` flag, a human must:**
1. Open the page and confirm the decline looks real (not a seasonal dip, a tracking gap, or a one-off traffic spike inflating the "prev" window).
2. Check whether a technical issue (deindexing, broken redirect, a competitor outranking it) is the actual cause — a content refresh won't fix a technical problem.
3. Confirm the page still matches a topic worth the editorial time — some `high_demand_at_risk` pages may have lost relevance entirely (the topic itself faded), which refreshing content won't reverse.

**What should NEVER be automated from this queue directly:**
- **Auto-publishing any content change.** The model flags *what* might be worth reviewing, never *what to write*.
- **Auto-deprioritizing or removing pages** based on a low score — a low `decline_probability` is not evidence a page is healthy, only that it doesn't match this sample's decline pattern.
- **Client-facing reporting of `decline_probability` as a certainty.** It's a ranking signal at ~0.76 precision@50, not a diagnosis.
- **Cross-client comparison of raw scores.** The model was validated on held-out *clients*, but scores across very different clients (site size, niche, baseline traffic) aren't necessarily on a comparable scale — rank within a client, don't compare across clients.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# A concrete illustration of "check before acting": how often does a flagged page (refresh_review)
# have near-zero remaining demand, where a refresh's upside is real but small?
flagged = ranked[ranked["action"] == "refresh_review"]
low_demand_flagged = flagged[flagged["search_volume"] < flagged["search_volume"].median()]
print(f"of {len(flagged):,} flagged pages, {len(low_demand_flagged):,} "
      f"({len(low_demand_flagged)/len(flagged):.0%}) sit below the flagged group's own median search volume")
print("-- these are exactly the rows where a human should sanity-check topic relevance before spending review time")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision drift:** if a monthly spot-check of the top 50 flagged pages shows precision@50 meaningfully below the ~0.76 measured in ML-08/ML-09 (say, below 0.6 on a fresh sample), the model needs review before the queue is trusted further.
- **Reason-code distribution shift:** if `reason_code` counts drift heavily toward categories that were thin in this run (`stale_and_unattended` was only 8 of ~9,600 test rows here) becoming dominant, that signals the underlying data or client mix has changed enough that the model's learned weighting may no longer fit.
- **New client onboarding:** per ML-09's own before/after finding, scores generalize with a real but limited gap (grouped vs. random split cost ~0.10 precision@50) — a brand-new client should be treated as lower-confidence territory until enough of their own history accumulates, not scored with full confidence from day one.
- **Time decay:** this was trained on one static 90-day cross-section; if the warehouse data (ML-04 onward) shows the underlying `trend_direction` base rate shifting meaningfully from the ~54% observed here, retraining — not just rescoring — is the right move, since the model's calibration assumes something close to that base rate.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"current base rate this run was trained/validated on: {y.mean():.1%}")
print(f"reason codes with n < 20 in this run (watch these for distribution shift):")
print(ranked["reason_code"].value_counts()[ranked["reason_code"].value_counts() < 20])

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["content_id", "client_id", "decline_probability", "reason_code", "action",
               "archetype", "click_equivalent_value", "avg_position", "days_since_last_update",
               "impressions_90d", "search_volume"]
ranked[export_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

metrics = {
    "model": "Logistic Regression (grouped-by-client split, ML-08/ML-09)",
    "queue_size": int(len(ranked)),
    "n_flagged_refresh_review": int((ranked["action"] == "refresh_review").sum()),
    "precision_at_50_honest_split": 0.76,
    "reason_code_counts": ranked["reason_code"].value_counts().to_dict(),
    "archetype_counts": ranked["archetype"].value_counts().to_dict(),
    "total_click_equivalent_value_flagged": round(float(flagged["click_equivalent_value"].sum()), 2),
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

fig, ax = plt.subplots(figsize=(7, 4))
ranked["reason_code"].value_counts().plot(kind="barh", ax=ax)
ax.set_xlabel("count")
ax.set_title("Reason codes across the held-out action queue")
plt.tight_layout()
plt.savefig("work/figures/w07_reason_code_distribution.png", dpi=150)
plt.close()

print("wrote work/outputs/content_action_playbook.csv (git-ignored, regenerated on every run)")
print("wrote work/outputs/w07_playbook_metrics.json (committed -- the paper's receipts)")
print("wrote work/figures/w07_reason_code_distribution.png (committed -- reused in the paper)")
print()
print(json.dumps(metrics, indent=2)[:600])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.